# Outras estatísticas dos times
## Etapa de construção de tabela gold

Ao menos 1 jogo do Mirassol não aparece estatistica para Faltas, assim tanto o número de faltas quanto a quantidade de partidas que tiveram dados de faltas pode ser menor que o real.

In [0]:
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.window import Window

In [0]:
df_statistics = spark.read.table("apifootball.silver.statistics")

In [0]:
df_home = (
    df_statistics
    .select(
        col("match_id"),
        col("statistic"),
        col("hometeam_id").alias("team_id"),
        col("hometeam_name").alias("team_name"),
        col("home_value").alias("value")
    )
)

df_away = (
    df_statistics
    .select(
        col("match_id"),
        col("statistic"),
        col("awayteam_id").alias("team_id"),
        col("awayteam_name").alias("team_name"),
        col("away_value").alias("value")
    )
)

df_union = df_home.unionByName(df_away)

In [0]:
df_corners = (
    df_union
    .filter(lower(col("statistic")) == 'corners')
    .groupBy("team_id","team_name")
    .agg(
        sum("value").alias("corners"),
        countDistinct("match_id").alias("matches"),
        round(sum("value") / countDistinct("match_id"), 2).alias("avg_corners")
    )
)

In [0]:
df_fouls = (
    df_union
    .filter(lower(col("statistic")) == 'fouls')
    .groupBy("team_id","team_name")
    .agg(
        sum("value").alias("fouls"),
        countDistinct("match_id").alias("matches"),
        round(sum("value") / countDistinct("match_id"), 2).alias("avg_fouls")
    )
)

In [0]:
df_offsides = (
    df_union
    .filter(lower(col("statistic")) == 'offsides')
    .groupBy("team_id","team_name")
    .agg(
        sum("value").alias("offsides"),
        countDistinct("match_id").alias("matches"),
        round(sum("value") / countDistinct("match_id"), 2).alias("avg_offsides")
    )
)

In [0]:
df_penalty = (
    df_union
    .filter(lower(col("statistic")) == 'penalty')
    .groupBy("team_id","team_name")
    .agg(
        sum("value").alias("penalties"),
        countDistinct("match_id").alias("matches"),
        round(sum("value") / countDistinct("match_id"), 2).alias("avg_penalties")
    )
)

In [0]:
df_team_statistics = (
    df_corners.alias("c")
    .join(df_fouls.alias("f"), col("f.team_id") == col("c.team_id"), "left")
    .join(df_offsides.alias("o"), col("o.team_id") == col("c.team_id"), "left")
    .join(df_penalty.alias("p"), col("p.team_id") == col("c.team_id"), "left")
    .select(
        col("c.team_id"),
        col("c.team_name"),
        col("c.corners"),
        col("c.avg_corners"),
        col("f.fouls"),
        col("f.avg_fouls"),
        col("o.offsides"),
        col("o.avg_offsides"),
        col("p.penalties"),
        col("p.avg_penalties"),
        expr('current_timestamp() - INTERVAL 3 HOURS').alias('datetime_processing_br')
        )
).orderBy(col("team_name"))

In [0]:
df_team_statistics.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("apifootball.gold.team_statistics")

In [0]:
#spark.read.table("apifootball.gold.team_statistics").display()